# Challenge 02 — Inteligencia Geo-Temporal y de Redes

**Optimización de Activos Críticos: TechLogistics S.A.**

EAFIT · Maestría en Ciencia de Datos y Analítica · Periodo 2026-1

Este notebook desarrolla el análisis multidimensional (geoespacial, series de tiempo, procesamiento de señales y grafos) solicitado en el Challenge 03.

Importación de librerías

In [1]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.arima.model import ARIMA
from scipy import stats, signal

## Fase 1: Data Understanding y Geo-Visualización

### Tarea 1: Exploración Geo-Temporal

#### Diccionario de variables de agro_clean.csv

Extraído de Lecture_03_dictionary.pdf

| Variable | Descripción | Naturaleza / Propósito |
|---|---|---|
| `Agro_1` – `Agro_3` | Variables hídricas: Humedad, Evapotranspiración y Humedad Relativa (RH) | Correlación alta entre sí. Estacionarias, **I(0)** |
| `Agro_4` | Radiación PAR (fotosintéticamente activa) | Cíclica — ciclo día/noche |
| `Agro_5` – `Agro_7` | Índices bióticos: **NDVI** y Biomasa | **No estacionarias, I(1)** — tienden a mostrar deriva (drift) en el tiempo |
| `Agro_8` – `Agro_10` | Suelo y viento | Estacionarias — ruido blanco con media constante |
| `Latitude` | Eje Y espacial | Posicionamiento del sensor en el oriente antioqueño |
| `Longitude` | Eje X espacial | Coordenadas decimales, usadas para clustering geoespacial |
| `Source_Node` | ID del sensor (nodo de origen) | Topología de la red mesh del cultivo |
| `Target_Node` | ID del gateway (nodo de destino) | Concentrador de datos al que reporta el sensor |

Carga del dataset y exploración rápida para confirmar tipos de dato, ausencia de nulos y los rangos de `Latitude`/`Longitude`.

In [2]:
agro = pd.read_csv("../data/agro_clean.csv")

print("Dimensiones:", agro.shape)
agro.info()
agro[["Latitude", "Longitude", "Agro_1", "Agro_5"]].describe()

Dimensiones: (2000, 14)
<class 'pandas.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 14 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Agro_1       2000 non-null   float64
 1   Agro_2       2000 non-null   float64
 2   Agro_3       2000 non-null   float64
 3   Agro_4       2000 non-null   float64
 4   Agro_5       2000 non-null   float64
 5   Agro_6       2000 non-null   float64
 6   Agro_7       2000 non-null   float64
 7   Agro_8       2000 non-null   float64
 8   Agro_9       2000 non-null   float64
 9   Agro_10      2000 non-null   float64
 10  Latitude     2000 non-null   float64
 11  Longitude    2000 non-null   float64
 12  Source_Node  2000 non-null   int64  
 13  Target_Node  2000 non-null   int64  
dtypes: float64(12), int64(2)
memory usage: 218.9 KB


,Latitude,Longitude,Agro_1,Agro_5
count,2000.000000,2000.000000,2000.000000,2000.000000
mean,6.202395,-75.397948,60.469678,1.158403
std,0.058344,0.057330,10.519590,0.379320
min,6.100010,-75.499898,43.554852,0.417623
25%,6.153419,-75.447321,49.997291,0.856683
50%,6.204413,-75.397947,61.202955,1.070661
75%,6.252968,-75.349042,70.809795,1.429127
max,6.299980,-75.300241,76.082501,2.321211


In [3]:
agro.head()

,Agro_1,Agro_2,Agro_3,Agro_4,Agro_5,Agro_6,Agro_7,Agro_8,Agro_9,Agro_10,Latitude,Longitude,Source_Node,Target_Node
0,60.248357,47.797447,65.395554,0.000000,0.478718,2.498349,10.000000,6.432151,1.258741,4.663639,6.203680,-75.400366,4,17
1,60.080940,48.076702,66.143237,25.002075,0.467100,2.473166,10.006672,6.469450,1.106051,4.683748,6.282310,-75.474525,14,21
2,60.623974,48.002378,66.342755,49.941596,0.449259,2.464547,10.013349,6.440262,1.183610,8.353141,6.165190,-75.476937,14,22
3,61.211672,48.267738,66.826015,74.756163,0.439299,2.500284,10.020030,6.511042,1.197917,6.213966,6.131334,-75.468152,8,25
4,60.483063,47.912028,65.703353,99.383693,0.436016,2.564177,10.026716,6.619718,1.200795,8.720976,6.241746,-75.349805,5,26


Observamos que la data esta completa y sin atípicos para cada una de las variables. Los rangos los usaremos para el zoom del mapa

Le pedimos a la IA el gráfico con el siguiente prompt:

***Usando plotly express grafica un scatter_mapbox para ver la ubicación de los sensores en el oriente antioqueño. Codifica el color de los puntos según el NVDI usando Agro_5 y el tamaño según la humedad usando Agro_1. Utiliza el rango para definir el zoom adecuado para enfocar el mapa en el oriente antioqueño***

In [4]:
centro_mapa = {"lat": agro["Latitude"].mean(), "lon": agro["Longitude"].mean()}

fig_mapa = px.scatter_mapbox(
    agro,
    lat="Latitude",
    lon="Longitude",
    color="Agro_5",
    size="Agro_1",
    color_continuous_scale="RdYlGn",
    size_max=12,
    zoom=10,
    center=centro_mapa,
    mapbox_style="carto-positron",
    hover_data=["Source_Node", "Target_Node", "Agro_1", "Agro_5"],
    labels={"Agro_5": "NDVI", "Agro_1": "Humedad"},
    title="Sensores agroindustriales — Oriente Antioqueño (color = NDVI, tamaño = Humedad)",
)
fig_mapa.update_layout(margin={"r": 0, "t": 40, "l": 0, "b": 0})
fig_mapa.show()

C:\Users\DanielSantiagoCadavi\AppData\Local\Temp\ipykernel_5132\2944694611.py:3: DeprecationWarning: *scatter_mapbox* is deprecated! Use *scatter_map* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/
  fig_mapa = px.scatter_mapbox(


Al revisar el mapa no se identifica un patrón espacial claro de clustering para el NDVI los puntos con `Agro_5` bajo aparecen dispersos de manera prácticamente uniforme sobre toda el área cubierta por los sensores, sin concentrarse en una subzona particular. Esto se presenta también para todos los niveles de NDVI.

Por tanto, no hay evidencia de clustering geoespacial de biomasa baja. La variabilidad de NDVI que se ve en el mapa puede deberse a otros factores como una deriva temporal (posible estacionalidad o degradación del cultivo) y no a la ubicación geográfica del sensor.

#### Exploración complementaria para la Pregunta de Validación 4: 
¿Cómo influye la posición geográfica en la varianza de la señal
capturada?

Para responder esta pregunta, le hacemos la siguiente solicitud a la IA:

***Divide el área de estudio en 4 cuadrantes usando la mediana de cada coordenada como líneas de corte: Norte/Sur y Este/Oeste. Para cada zona calcula la varianza de `Agro_1` (Humedad) y `Agro_5` (NDVI) y comparalas mediante una tabla y un gráfico de barras, en 2 subplot diferente cada variable***

In [5]:
lat_zona = pd.qcut(agro["Latitude"], 2, labels=["Sur", "Norte"])
lon_zona = pd.qcut(agro["Longitude"], 2, labels=["Oeste", "Este"])
agro["Zona"] = lat_zona.astype(str) + "-" + lon_zona.astype(str)

varianza_por_zona = (
    agro.groupby("Zona")[["Agro_1", "Agro_5"]]
    .var()
    .rename(columns={"Agro_1": "Var(Humedad)", "Agro_5": "Var(NDVI)"})
)

fig_var = make_subplots(
    rows=2,
    cols=1,
    subplot_titles=("Varianza de Humedad (Agro_1)", "Varianza de NDVI (Agro_5)"),
)
fig_var.add_trace(
    go.Bar(
        x=varianza_por_zona.index,
        y=varianza_por_zona["Var(Humedad)"],
        marker_color="#1f77b4",
        name="Var(Humedad)",
    ),
    row=1,
    col=1,
)
fig_var.add_trace(
    go.Bar(
        x=varianza_por_zona.index,
        y=varianza_por_zona["Var(NDVI)"],
        marker_color="#2ca02c",
        name="Var(NDVI)",
    ),
    row=2,
    col=1,
)
fig_var.update_yaxes(title_text="Varianza", row=1, col=1)
fig_var.update_yaxes(title_text="Varianza", row=1, col=2)
fig_var.update_layout(
    title_text="Varianza de Humedad y NDVI por zona geográfica", showlegend=False
)
fig_var.show()

varianza_por_zona

,Var(Humedad),Var(NDVI)
Zona,,
Norte-Este,112.833401,0.137334
Norte-Oeste,111.653211,0.154136
Sur-Este,106.568089,0.148504
Sur-Oeste,111.905746,0.135431


De la gráfica de barras y la tabla podemos ver que:
- La varianza de `Agro_1` (Humedad) varía poco entre zonas (106 a 112, dispersión relativa de 6% aproximadamente)
- La varianza de `Agro_5` (NDVI) muestra una dispersión algo mayor entre zonas en términos porcentuales (0.135 a 0.154, 13% aprox).

En ambos casos las diferencias son pequeñas y ninguna zona se diferencia notablemente de las demás. Esto sugiere que la posición geográfica no tiene influencia sobre la varianza de las señales capturadas: los sensores registran una variabilidad relativamente homogénea sin importar el cuadrante del oriente antioqueño en el que estén ubicados. 

Esto es coherente con el análisis del mapa: en este dataset, la varianza de la señal parece explicarse más por otros factores que por la geografía.

### Tarea 2: Análisis de Estacionariedad y Windowing

#### Diccionario de variables de ener_clean.csv

Extraído de Lecture_03_dictionary.pdf

| Variable | Descripción | Naturaleza / Propósito |
|---|---|---|
| `Ener_1` – `Ener_3` | Mercado Spot: Demanda, Precio y Temperatura | Correlación alta entre sí |
| `Ener_4` | Generación Eólica | Cíclica compleja (intermitencia estocástica) |
| `Ener_5` – `Ener_7` | Factores macro: Costo de Gas y Emisiones CO2 | **No estacionarias** |
| `Ener_8` – `Ener_10` | Calidad de potencia: Frecuencia, Voltaje y Factor de Potencia | **Estacionarias** |
| `Latitude` | Ubicación del nodo | Coordenadas de subestaciones a nivel nacional (Colombia) |
| `Longitude` | Ubicación del nodo | Permite análisis de flujos de potencia regionales |
| `Source_Node` | Subestación | Nodo de generación o transformación primaria |
| `Target_Node` | Nodo de carga | Punto de consumo o bus de distribución |

Carga del dataset y exploración rápida para confirmar tipos de dato, ausencia de nulos y rangos

In [6]:
ener = pd.read_csv("../data/ener_clean.csv")

print("Dimensiones:", ener.shape)
ener.info()
ener.describe()

Dimensiones: (2000, 14)
<class 'pandas.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 14 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Ener_1       2000 non-null   float64
 1   Ener_2       2000 non-null   float64
 2   Ener_3       2000 non-null   float64
 3   Ener_4       2000 non-null   float64
 4   Ener_5       2000 non-null   float64
 5   Ener_6       2000 non-null   float64
 6   Ener_7       2000 non-null   float64
 7   Ener_8       2000 non-null   float64
 8   Ener_9       2000 non-null   float64
 9   Ener_10      2000 non-null   float64
 10  Latitude     2000 non-null   float64
 11  Longitude    2000 non-null   float64
 12  Source_Node  2000 non-null   int64  
 13  Target_Node  2000 non-null   int64  
dtypes: float64(12), int64(2)
memory usage: 218.9 KB


,Ener_1,Ener_2,Ener_3,Ener_4,Ener_5,Ener_6,Ener_7,Ener_8,Ener_9,Ener_10,Latitude,Longitude,Source_Node,Target_Node
count,2000.000000,2000.000000,2000.000000,2000.000000,2000.000000,2000.000000,2000.000000,2000.000000,2000.000000,2000.000000,2000.000000,2000.000000,2000.000000,2000.000000
mean,99.118764,148.580985,24.751477,39.576716,15.587930,903.963850,519.979570,59.999702,110.013818,0.950203,7.749894,-74.472195,109.596500,224.908500
std,14.440028,21.897587,3.614629,14.152858,6.473704,57.405493,11.585182,0.020124,0.499752,0.010085,1.855723,1.441678,5.804096,14.262883
min,74.913525,103.806723,18.318555,10.366132,4.370024,802.709465,498.620530,59.933707,108.415106,0.910384,4.503435,-76.999355,100.000000,200.000000
25%,84.913778,128.346756,21.230491,29.591519,10.303586,856.425516,510.016803,59.986516,109.673423,0.943623,6.171752,-75.691385,105.000000,212.000000
50%,98.212758,147.090345,24.450424,39.500832,15.171830,906.005350,520.099689,59.999748,110.008317,0.950002,7.739210,-74.463218,110.000000,225.000000
75%,113.373378,168.991990,28.366448,49.841833,21.583653,955.940098,530.024822,60.013246,110.343654,0.956695,9.352822,-73.232738,115.000000,237.000000
max,123.783193,191.088326,31.563383,69.633831,26.168797,1000.185140,541.256191,60.081989,111.888122,0.984070,10.993939,-72.000692,119.000000,249.000000


Observamos que la data esta completa y con valores aparentemente lógicos para cada una de las variables.

Para aplicar el test ADF a las series de energía le enviamos el siguiente prompt a la IA:

***Enuncia y aplica el test ADF a las series de energía. Para las series no estacionarias, aplica una ventana móvil de 50 registros para calcular la media y vairanza***

#### Test de Dickey-Fuller Aumentado (ADF)

El test ADF evalúa si una serie de tiempo tiene una **raíz unitaria**, es decir, si es no estacionaria.

- **Hipótesis nula (H0):** la serie tiene raíz unitaria → es **no estacionaria** (su media y/o varianza cambian en el tiempo).
- **Hipótesis alternativa (H1):** la serie no tiene raíz unitaria → es **estacionaria**.
- **p-value:** probabilidad de observar un estadístico ADF tan extremo como el calculado si H0 fuera cierta. Un p-value bajo es evidencia en contra de H0.
- **Criterio de decisión (umbral 0.05):** si `p-value < 0.05` rechazamos H0 y concluimos que la serie es estacionaria; si `p-value >= 0.05` no hay evidencia suficiente para rechazar H0 y tratamos la serie como no estacionaria.

Aplicamos `adfuller` (de `statsmodels.tsa.stattools`) a cada una de las columnas `Ener_1` a `Ener_10`, guardando el estadístico ADF y el p-value de cada una. Con el criterio anterior armamos una tabla resumen que indica si cada serie es estacionaria o no, la cual usamos después para decidir a qué series les aplicamos el análisis de ventana móvil.

In [7]:
cols_ener = [f"Ener_{i}" for i in range(1, 11)]

resultados_adf = []
for col in cols_ener:
    estadistico, p_value, *_ = adfuller(ener[col], autolag="AIC")
    resultados_adf.append(
        {
            "Serie": col,
            "Estadistico_ADF": estadistico,
            "p_value": p_value,
            "Estacionaria": p_value < 0.05,
        }
    )

resumen_adf = pd.DataFrame(resultados_adf)
resumen_adf

,Serie,Estadistico_ADF,p_value,Estacionaria
0,Ener_1,-1.866891e+00,0.347793,False
1,Ener_2,-1.479135e+00,0.543807,False
2,Ener_3,-1.928441e+00,0.318802,False
3,Ener_4,-1.336821e+10,0.000000,True
4,Ener_5,-3.477077e-01,0.918470,False
5,Ener_6,9.526991e-01,0.993741,False
6,Ener_7,-4.244451e-01,0.905937,False
7,Ener_8,-4.603634e+01,0.000000,True
8,Ener_9,-4.535819e+01,0.000000,True
9,Ener_10,-4.371511e+01,0.000000,True


Según la tabla anterior, el test ADF nos permite concluir que:
- `Ener_1`, `Ener_2`, `Ener_3`, `Ener_5`, `Ener_6` y `Ener_7` resultan no estacionarias (p-value >= 0.05), 
- Mientras que `Ener_4`, `Ener_8`, `Ener_9` y `Ener_10` sí lo son.


Esto coincide con el diccionario de datos: `Ener_5`, `Ener_6` y `Ener_7` están etiquetadas como no estacionarias y `Ener_8`, `Ener_9` y `Ener_10` como estacionarias.

Para las no estacionarias calculamos la **media móvil** y la **desviación estándar móvil** con una ventana de 50 registros y las graficamos superpuestas a la serie original.

Para esto le pedimos a la IA lo siguiente:

***Para las series no estacionarias, calcula la media móvil y desviación estándar con una ventana de 50 registros. Grafícalas superpuesta a la serie original de modo que pueda ver bien su comportamiento***

Como la escala de la desviación estándar móvil es mucho menor que la de la serie original (por ejemplo, `Ener_6` varía entre ~800 y ~1000 mientras su desviación móvil varía entre ~0.3 y ~2.7), separamos cada serie en dos paneles: uno con la serie original + su media móvil (misma escala) y otro con su desviación estándar móvil (escala propia).

In [8]:
no_estacionarias = resumen_adf.loc[~resumen_adf["Estacionaria"], "Serie"].tolist()
ventana = 50

fig_rolling = make_subplots(
    rows=len(no_estacionarias),
    cols=2,
    subplot_titles=[
        titulo
        for col in no_estacionarias
        for titulo in (
            f"{col}: serie original (gris) y media móvil (azul)",
            f"{col}: desviación estándar móvil (rojo)",
        )
    ],
    vertical_spacing=0.06,
    horizontal_spacing=0.08,
)

for i, col in enumerate(no_estacionarias, start=1):
    serie = ener[col]
    media_movil = serie.rolling(ventana).mean()
    std_movil = serie.rolling(ventana).std()

    fig_rolling.add_trace(
        go.Scatter(y=serie, name="Serie original", line=dict(color="#7f7f7f", width=1)),
        row=i,
        col=1,
    )
    fig_rolling.add_trace(
        go.Scatter(y=media_movil, name="Media móvil (w=50)", line=dict(color="#1f77b4", width=2)),
        row=i,
        col=1,
    )
    fig_rolling.add_trace(
        go.Scatter(y=std_movil, name="Desv. estándar móvil (w=50)", line=dict(color="#d62728", width=2)),
        row=i,
        col=2,
    )

fig_rolling.update_annotations(font_size=12)
fig_rolling.update_layout(
    height=260 * len(no_estacionarias),
    title=dict(
        text="Media y desviación estándar móvil (ventana = 50 registros) — series no estacionarias",
        x=0.5,
        xanchor="center",
    ),
    showlegend=False,
    margin=dict(t=70, b=40),
)
fig_rolling.show()

En las gráficas se observa media móvil no constante en las seis series, aunque con distintos patrones: 
- `Ener_5`, `Ener_6` y `Ener_7` muestran una tendencia (drift) que se mueve hacia un mismo sentido consistentemente en toda la serie.
- `Ener_1`, `Ener_2` y `Ener_3` parecen tener un cambio de nivel (sube y luego vuelve a bajar) a lo largo de la serie. 

En ambos casos la media no es constante en el tiempo, lo cual es consistente con no haber rechazado H0 en el test ADF.

#### Ener_5 (Costo del Gas): ¿Random Walk puro o con Drift?

Graficamos la serie `Ener_5` y su primera diferencia. Si `Ener_5` es un random walk, su primera diferencia debería comportarse como ruido blanco con media cero. Si además existe un **drift** (deriva constante), la media de la diferencia será significativamente distinta de cero. 

Para esto le pedimos a la IA:

***Elabora un código que me ayude a comprobar tanto gráfica como analíticamente si Ener_5 es un random walk o no***

Para decidir entre ambos casos aplicamos un t-test de una muestra (`scipy.stats.ttest_1samp`) sobre la primera diferencia, con H0: media de la diferencia = 0. De paso, aplicamos también ADF sobre la diferencia para confirmar si con un solo grado de diferenciación se logra estacionariedad (I(1)).

In [9]:
serie_ener5 = ener["Ener_5"]
diff_ener5 = serie_ener5.diff().dropna()

fig_ener5 = make_subplots(
    rows=2,
    cols=1,
    subplot_titles=("Ener_5: serie original (Costo del Gas)", "Ener_5: primera diferencia"),
    vertical_spacing=0.12,
)
fig_ener5.add_trace(go.Scatter(y=serie_ener5, name="Ener_5", line=dict(color="#1f77b4")), row=1, col=1)
fig_ener5.add_trace(go.Scatter(y=diff_ener5, name="Diferencia", line=dict(color="#ff7f0e")), row=2, col=1)
fig_ener5.update_layout(height=600, title_text="Ener_5 — Serie original vs. primera diferencia", showlegend=False)
fig_ener5.show()

media_diff = diff_ener5.mean()
t_stat, p_value_ttest = stats.ttest_1samp(diff_ener5, popmean=0.0)
adf_diff_stat, adf_diff_p, *_ = adfuller(diff_ener5, autolag="AIC")

print(f"Media de la primera diferencia: {media_diff:.5f}")
print(f"t-test (H0: media = 0): t = {t_stat:.3f}, p-value = {p_value_ttest:.2e}")
print(f"ADF sobre la primera diferencia: estadistico = {adf_diff_stat:.3f}, p-value = {adf_diff_p:.2e}")

Media de la primera diferencia: 0.01056
t-test (H0: media = 0): t = 4.664, p-value = 3.31e-06
ADF sobre la primera diferencia: estadistico = -43.656, p-value = 0.00e+00


Como vimos anteriormente, el test ADF sobre `Ener_5` no rechazó H0, por lo que la serie es no estacionaria. Su primera diferencia, en cambio, sí resulta estacionaria (ADF con p-value ≈ 0), lo que confirma que `Ener_5` es integrada de orden 1 (I(1)). 

El t-test sobre la diferencia da `t ≈ 4.66` con `p-value ≈ 3.3e-06` (< 0.05), por lo que rechazamos H0: media de la diferencia = 0. 

La primera diferencia tiene una media positiva pequeña pero significativa (0.0106 aproximadamente), y la serie original crece de forma sostenida (de 5 a 26 en todo el rango de datos aproximadamente). Por lo tanto, `Ener_5` se comporta como un **Random Walk con Drift** y no como un random walk puro: la deriva constante y positiva explica el crecimiento sostenido del Costo del Gas.

#### Pregunta de Validación 1: 

¿por qué no es válido aplicar correlación de Pearson sobre una serie con tendencia?

La correlación de Pearson asume, implícitamente, que las series involucradas tienen una relación estadística estable en el tiempo (medias y varianzas constantes). Cuando dos series comparten una tendencia, como el NDVI (Agro_5) o el Costo del Gas (Ener_5), ambas crecen o decrecen de forma sostenida simplemente por el paso del tiempo, y no porque exista una relación causal o estructural entre ellas. Esto ocasiona que el coeficiente de Pearson puede salir muy alto (positivo o negativo) solo porque ambas series se mueven en la misma dirección a lo largo del tiempo, sin que exista un vínculo real entre sus variaciones de corto plazo.

La forma correcta de analizar la relación entre dos series de este tipo es primero eliminar la tendencia hasta lograr estacionariedad, y ahí sí calcular la correlación.

## Fase 2: Procesamiento de Señales y Filtrado

### Tarea 3: Análisis Espectral (FFT) y Espectrogramas

Para este apartado usamos el siguiente prompt:

***Sobre Ener_4 extrae la densidad espectral de potencia mediante FFT (definela) y aparte haz un espectrograma de la serie clena vs. noise para determinar el rango de frecuencia del ruido***

#### FFT, densidad espectral de potencia (PSD) y espectrograma

- **FFT (Transformada Rápida de Fourier):** descompone una señal en el tiempo en sus componentes de frecuencia, indicando qué tan presente está cada frecuencia en la serie completa.
- **PSD (densidad espectral de potencia):** se obtiene a partir de la magnitud de la FFT al cuadrado (`|FFT|² / N`). Muestra en qué frecuencias se concentra la energía de la señal, promediada sobre toda la serie.
- **Espectrograma:** aplica la FFT sobre ventanas cortas y solapadas de la señal (en vez de sobre toda la serie de una vez), y apila los resultados en el tiempo. Así se obtiene cómo cambia el contenido de frecuencia a lo largo de la serie, algo que la PSD (un solo promedio global) no puede mostrar.

Para una señal como la Generación Eólica (`Ener_4`), descrita en el diccionario como "cíclica compleja con intermitencia estocástica", la PSD nos dice si existe algún ciclo dominante (ráfagas de viento recurrentes) y el espectrograma nos permite ver si ese contenido de frecuencia se mantiene estable en el tiempo o cambia (intermitencia). Además, comparando el espectrograma de la versión `clean` contra la `noise` podemos ubicar en qué banda de frecuencias se concentra el ruido inyectado.

Calculamos la FFT de `Ener_4` (clean) con `numpy.fft.rfft` (la versión para señales reales, que solo devuelve las frecuencias positivas) y derivamos la PSD como `|FFT|² / N`. Asumimos una frecuencia de muestreo `fs = 1` muestra/registro, ya que no hay una marca de tiempo explícita en el dataset. Graficamos la PSD en escala logarítmica en el eje Y porque la energía se concentra en unas pocas frecuencias bajas y se dispersa en varios órdenes de magnitud.

In [10]:
fs = 1.0  # 1 muestra / registro
ener4_clean = ener["Ener_4"].values
N = len(ener4_clean)

fft_vals = np.fft.rfft(ener4_clean)
frecuencias = np.fft.rfftfreq(N, d=1 / fs)
psd = (np.abs(fft_vals) ** 2) / N

fig_psd = px.line(
    x=frecuencias[1:],
    y=psd[1:],
    log_y=True,
    labels={"x": "Frecuencia (ciclos/registro)", "y": "PSD (escala log)"},
    title="Ener_4 (clean) — Densidad espectral de potencia (FFT)",
)
fig_psd.show()

frecuencia_pico = frecuencias[1:][np.argmax(psd[1:])]
print(f"Frecuencia dominante: {frecuencia_pico:.4f} ciclos/registro")
print(f"Periodo asociado: {1 / frecuencia_pico:.0f} registros")

Frecuencia dominante: 0.0035 ciclos/registro
Periodo asociado: 286 registros


La PSD muestra un pico dominante alrededor de 0.003-0.0035 ciclos/registro, es decir, un ciclo de aproximadamente 300 registros. El resto de la energía cae rápido hacia frecuencias más altas, lo cual es consistente con una señal cíclica de periodo largo (ráfagas de viento que se repiten cada ~300 registros) más componentes estocásticos de menor energía superpuestos.

Cargamos `ener_noise.csv` y calculamos el espectrograma de `Ener_4` para las versiones `clean` y `noise` con `scipy.signal.spectrogram` (ventanas de 128 registros, solapamiento de 96). Convertimos la potencia a decibelios (`10*log10`) porque, igual que con la PSD, la energía varía en varios órdenes de magnitud. Graficamos ambos espectrogramas lado a lado compartiendo la misma escala de color (mismo `cmin`/`cmax`) para que la comparación visual entre clean y noise sea justa.

In [11]:
ener_noise = pd.read_csv("../data/ener_noise.csv")

nperseg, noverlap = 128, 96

f_clean, t_clean, Sxx_clean = signal.spectrogram(ener["Ener_4"], fs=fs, nperseg=nperseg, noverlap=noverlap)
f_noise, t_noise, Sxx_noise = signal.spectrogram(ener_noise["Ener_4"], fs=fs, nperseg=nperseg, noverlap=noverlap)

Sxx_clean_db = 10 * np.log10(Sxx_clean + 1e-12)
Sxx_noise_db = 10 * np.log10(Sxx_noise + 1e-12)

zmin = min(Sxx_clean_db.min(), Sxx_noise_db.min())
zmax = max(Sxx_clean_db.max(), Sxx_noise_db.max())

fig_spec = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=("Ener_4 clean", "Ener_4 noise"),
    shared_yaxes=True,
)
fig_spec.add_trace(go.Heatmap(x=t_clean, y=f_clean, z=Sxx_clean_db, coloraxis="coloraxis"), row=1, col=1)
fig_spec.add_trace(go.Heatmap(x=t_noise, y=f_noise, z=Sxx_noise_db, coloraxis="coloraxis"), row=1, col=2)

fig_spec.update_layout(
    coloraxis=dict(colorscale="Viridis", cmin=zmin, cmax=zmax, colorbar=dict(title="Potencia (dB)")),
    title=dict(text="Ener_4 — Espectrograma: clean vs. noise (misma escala de color)", x=0.5, xanchor="center"),
    height=450,
)
fig_spec.update_xaxes(title_text="Tiempo (registros)")
fig_spec.update_yaxes(title_text="Frecuencia (ciclos/registro)", row=1, col=1)
fig_spec.show()

#### SNR real entre Ener_4 clean y noise

Con la definición del diccionario de datos, `SNR_dB = 10*log10(var_signal / var_noise)`, donde `var_signal` es la varianza de la serie `clean` y `var_noise` es la varianza del ruido aislado, `noise - clean`. Comparamos el resultado contra el rango teórico mencionado en el enunciado (5-12 dB).

In [12]:
diferencia_ruido = ener_noise["Ener_4"] - ener["Ener_4"]

var_senal = ener["Ener_4"].var()
var_ruido = diferencia_ruido.var()
snr_db = 10 * np.log10(var_senal / var_ruido)

print(f"Varianza de la señal (clean): {var_senal:.3f}")
print(f"Varianza del ruido (noise - clean): {var_ruido:.3f}")
print(f"SNR estimado: {snr_db:.2f} dB")
print(f"¿Dentro del rango teórico [5, 12] dB?: {5 <= snr_db <= 12}")

Varianza de la señal (clean): 200.303
Varianza del ruido (noise - clean): 28.072
SNR estimado: 8.53 dB
¿Dentro del rango teórico [5, 12] dB?: True


#### Análisis: ¿en qué rango de frecuencias se concentra el ruido inyectado?

El SNR estimado de 8.53 dB cae dentro del rango teórico anunciado [5-12] dB.

En cuanto a la banda de frecuencias, al comparar los dos espectrogramas con la misma escala de color:

- En las frecuencias bajas (por debajo de 0.02-0.03 ciclos/registro, donde vimos el pico dominante de la PSD) ambos espectrogramas se ven parecidos: ahí manda la señal eólica real, y el ruido agregado es pequeño frente a la energía propia de la señal.
- En frecuencias medias y altas (desde 0.05 hasta el límite de Nyquist en 0.5 ciclos/registro) el espectrograma `clean` se ve prácticamente vacío/oscuro: la señal original casi no tiene energía ahí. Mientras que el de `noise` muestra energía visiblemente mayor y repartida de forma pareja en todo ese rango.

Esto es el comportamiento esperado de un ruido blanco: al ser incorrelacionado en el tiempo, reparte su energía de forma aproximadamente uniforme en todo el espectro. Como la señal limpia concentra casi toda su energía en unas pocas frecuencias bajas, el ruido termina siendo indetectable ahí, pero domina claramente el resto del espectro, que es donde más se nota visualmente en el espectrograma.

### Tarea 4: Filtrado y Reconstrucción

#### Filtro Butterworth paso-bajo

Un filtro Butterworth paso-bajo deja pasar las frecuencias por debajo de una **frecuencia de corte** (`fc`) y atenúa las que están por encima, con una respuesta lo más plana posible en la banda que conserva (sin rizado). Es una elección natural para suavizar el ruido de alta frecuencia inyectado en `Agro_3_noise`: si la señal real (Humedad Relativa) varía lentamente y el ruido blanco reparte su energía en todo el espectro (como vimos con `Ener_4` en la Tarea 3), basta con cortar las frecuencias donde ya no queda señal útil para eliminar buena parte del ruido sin distorsionar la forma de la serie.

- **Frecuencia de corte (`fc`):** se elige a partir de dónde la señal limpia deja de tener energía relevante en su espectro. La revisamos con la FFT de `Agro_3` (clean vs. noise) más abajo. Un `fc` muy bajo suaviza de más (se pierde señal real); uno muy alto deja pasar ruido.
- **Orden del filtro:** controla qué tan abrupta es la transición entre la banda que se conserva y la que se atenúa. Un orden mayor filtra más agresivamente cerca de `fc`, pero también introduce más desfase — por eso aplicamos el filtro con `filtfilt`, que filtra hacia adelante y hacia atrás y cancela ese desfase.

Para elegir `fc` calculamos la PSD (igual que en la Tarea 3) de `Agro_3` en `agro_clean.csv` y en `agro_noise.csv`, y las comparamos. La idea es identificar la frecuencia a partir de la cual la energía de la serie `clean` ya es despreciable frente a la de `noise` — ahí es donde conviene cortar.

In [13]:
agro_noise = pd.read_csv("../data/agro_noise.csv")

fs = 1.0
agro3_clean = agro["Agro_3"].values
agro3_noise = agro_noise["Agro_3"].values


def calcular_psd(x, fs=1.0):
    N = len(x)
    fft_vals = np.fft.rfft(x)
    freqs = np.fft.rfftfreq(N, d=1 / fs)
    return freqs, (np.abs(fft_vals) ** 2) / N


frec_clean, psd_clean = calcular_psd(agro3_clean, fs)
frec_noise, psd_noise = calcular_psd(agro3_noise, fs)

fc_elegida = 0.01

fig_psd_agro = go.Figure()
fig_psd_agro.add_trace(go.Scatter(x=frec_clean[1:], y=psd_clean[1:], name="Agro_3 clean", line=dict(color="#1f77b4")))
fig_psd_agro.add_trace(go.Scatter(x=frec_noise[1:], y=psd_noise[1:], name="Agro_3 noise", line=dict(color="#d62728")))
fig_psd_agro.add_vline(x=fc_elegida, line_dash="dash", line_color="gray", annotation_text=f"fc = {fc_elegida}")
fig_psd_agro.update_yaxes(type="log", title_text="PSD (escala log)")
fig_psd_agro.update_xaxes(title_text="Frecuencia (ciclos/registro)")
fig_psd_agro.update_layout(title="Agro_3 — PSD clean vs. noise")
fig_psd_agro.show()

Las dos PSD son prácticamente idénticas por debajo de ~0.01-0.02 ciclos/registro (ahí vive la señal real de Humedad Relativa), y a partir de ese punto la PSD de `noise` se queda muy por encima de la de `clean` — la de `clean` cae varios órdenes de magnitud mientras la de `noise` se mantiene alta y plana (el mismo comportamiento de ruido blanco que vimos con `Ener_4` en la Tarea 3). Por eso elegimos `fc = 0.01` ciclos/registro (periodo ≈ 100 registros): conserva casi toda la energía de la señal real y descarta la mayor parte del ruido.

Diseñamos el filtro con `scipy.signal.butter(orden, fc, btype="low", fs=fs)` y lo aplicamos sobre `Agro_3` de `agro_noise.csv` con `scipy.signal.filtfilt`, que aplica el filtro dos veces (adelante y atrás) para cancelar el desfase que introduciría un filtrado normal. Usamos orden 4, un valor estándar que da una transición razonablemente abrupta sin ser excesivo para una serie de solo 2000 registros.

In [14]:
orden = 4

b, a = signal.butter(orden, fc_elegida, btype="low", fs=fs)
agro3_filtrada = signal.filtfilt(b, a, agro3_noise)

print(f"Filtro Butterworth: orden={orden}, fc={fc_elegida} ciclos/registro")

Filtro Butterworth: orden=4, fc=0.01 ciclos/registro


Graficamos las tres versiones de `Agro_3` superpuestas — clean, noise y filtrada — para comparar visualmente qué tan cerca queda la señal filtrada de la original.

In [15]:
fig_comparacion = go.Figure()
fig_comparacion.add_trace(go.Scatter(y=agro3_noise, name="Noise", line=dict(color="#d62728", width=1)))
fig_comparacion.add_trace(go.Scatter(y=agro3_clean, name="Clean", line=dict(color="#7f7f7f", width=2)))
fig_comparacion.add_trace(
    go.Scatter(y=agro3_filtrada, name="Filtrada (Butterworth)", line=dict(color="#1f77b4", width=2))
)
fig_comparacion.update_layout(
    title="Agro_3 (Humedad Relativa) — Clean vs. Noise vs. Filtrada",
    xaxis_title="Registro",
    yaxis_title="Humedad Relativa",
)
fig_comparacion.show()

Calculamos el RMSE entre la serie filtrada y la clean, y lo comparamos contra el RMSE entre noise y clean (nuestra línea base sin filtrar), para cuantificar qué tanto se acerca el filtrado a recuperar la señal original.

In [16]:
def rmse(x, y):
    return float(np.sqrt(np.mean((np.asarray(x) - np.asarray(y)) ** 2)))


rmse_noise = rmse(agro3_noise, agro3_clean)
rmse_filtrada = rmse(agro3_filtrada, agro3_clean)

tabla_rmse = pd.DataFrame(
    {
        "Comparacion": ["Noise vs. Clean", "Filtrada vs. Clean"],
        "RMSE": [rmse_noise, rmse_filtrada],
        "Mejora_%_vs_noise": [0.0, (1 - rmse_filtrada / rmse_noise) * 100],
    }
)
tabla_rmse

,Comparacion,RMSE,Mejora_%_vs_noise
0,Noise vs. Clean,3.341434,0.000000
1,Filtrada vs. Clean,0.845432,74.698512


El filtrado reduce el RMSE frente a la señal clean de ~3.34 a ~0.85, una mejora de ~75%. El filtro Butterworth recupera de forma consistente la forma de la señal real, eliminando la mayor parte del ruido de alta frecuencia.

#### ¿El filtrado mejora la capacidad predictiva?

Ajustamos un modelo AR(2) simple (`statsmodels.tsa.arima.model.ARIMA`, orden `(2,0,0)`) sobre las tres versiones de `Agro_3` — clean, noise y filtrada — y comparamos los coeficientes estimados (`const`, `ar.L1`, `ar.L2`, `sigma2`). Un modelo de orden bajo es razonable aquí porque `Agro_3` es una variable hídrica I(0) según el diccionario de datos, con autocorrelación esperada de corto plazo.

In [17]:
orden_ar = (2, 0, 0)

coeficientes = {}
for nombre, serie in [("Clean", agro3_clean), ("Noise", agro3_noise), ("Filtrada", agro3_filtrada)]:
    modelo = ARIMA(serie, order=orden_ar).fit()
    coeficientes[nombre] = pd.Series(modelo.params, index=modelo.param_names)

tabla_coeficientes = pd.DataFrame(coeficientes).T
tabla_coeficientes

,const,ar.L1,ar.L2,sigma2
Clean,69.229328,0.503945,0.493949,7.724800e-01
Noise,66.496399,0.482834,0.474518,1.712035e+01
Filtrada,66.463858,1.998653,-0.998759,9.982818e-07


Los coeficientes por sí solos no bastan para hablar de "capacidad predictiva"; para eso evaluamos el error de pronóstico fuera de muestra. Separamos el 90% inicial como entrenamiento y el 10% final (200 registros) como prueba, y hacemos pronósticos de un paso adelante de forma iterativa: ajustamos el modelo una vez con el entrenamiento y, en cada paso, lo actualizamos con la observación real de esa misma serie (`.append(..., refit=False)`) antes de pronosticar el siguiente punto. En los tres casos comparamos el pronóstico contra el valor real de la serie **clean**, que es la que nos interesa recuperar.

In [18]:
n = len(agro3_clean)
n_train = int(n * 0.9)
n_test = n - n_train


def pronostico_un_paso(serie, n_train, n_test, orden=orden_ar):
    modelo = ARIMA(serie[:n_train], order=orden).fit()
    pronosticos = []
    for i in range(n_test):
        pronosticos.append(modelo.forecast(steps=1)[0])
        modelo = modelo.append([serie[n_train + i]], refit=False)
    return np.array(pronosticos)


resultados_forecast = []
for nombre, serie in [("Clean", agro3_clean), ("Noise", agro3_noise), ("Filtrada", agro3_filtrada)]:
    pronosticos = pronostico_un_paso(serie, n_train, n_test)
    resultados_forecast.append(
        {"Serie": nombre, "RMSE_pronostico_vs_clean": rmse(pronosticos, agro3_clean[n_train:])}
    )

tabla_forecast = pd.DataFrame(resultados_forecast)
tabla_forecast

C:\Users\DanielSantiagoCadavi\AppData\Local\Programs\Python\Python314\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


,Serie,RMSE_pronostico_vs_clean
0,Clean,0.983547
1,Noise,2.428981
2,Filtrada,0.964602


**Conclusión:**

- **Coeficientes:** el AR(2) ajustado sobre `clean` da `ar.L1 ≈ 0.50` y `ar.L2 ≈ 0.49` (suman ≈0.998, alta persistencia) con `sigma2 ≈ 0.77`. Sobre `noise`, los coeficientes autorregresivos (`ar.L1 ≈ 0.48`, `ar.L2 ≈ 0.47`) cambian relativamente poco frente a clean, pero `sigma2` se dispara a ≈17.1 — el ruido no altera demasiado la dinámica autorregresiva estimada, pero infla mucho la varianza residual del modelo. Sobre la serie **filtrada**, los coeficientes cambian de forma mucho más drástica (`ar.L1 ≈ 2.0`, `ar.L2 ≈ -1.0`, `sigma2 ≈ 0`): el filtro Butterworth suaviza tanto la señal que el AR(2) ajustado ya no corresponde al mismo tipo de proceso que el de `clean`, sino a una curva casi determinística.
- **Pronóstico:** con pronósticos de un paso adelante contra la clean real, el modelo ajustado sobre `noise` tiene el peor desempeño (RMSE ≈ 2.43 aprox.), más del doble que el ajustado sobre `clean` (RMSE ≈ 0.98 aprox.). El modelo ajustado sobre la serie **filtrada** recupera un desempeño muy cercano al de `clean` (RMSE ≈ 0.96 aprox.).

En conjunto, el filtrado sí mejora la capacidad predictiva (el RMSE de pronóstico vuelve a niveles de `clean`), aunque lo hace a costa de estimar una dinámica interna (coeficientes AR) distinta a la original: el filtro no "recupera" el proceso generador real, sino que produce una versión suavizada más fácil de predecir. Esto también es el insumo para la Pregunta de Validación 2 del checklist: el ruido de 5-12dB no distorsiona demasiado los coeficientes autorregresivos de corto plazo del ARMA frente a `clean`, pero sí dispara la varianza del error (`sigma2`) y degrada notablemente el pronóstico.

## Fase 3: Análisis de Grafos y Topología de Red

### Tarea 5: Construcción de la Red de Sensores/Subestaciones

## Fase 4: Modelado y Toma de Decisiones (CRISP-DM)

### P1: Causalidad y Redes

### P2: Optimización Geo-Agrónoma

### P3: Analítica Predictiva (ARIMAX)